### `make_tea(name, seconds)`

An **async** function that simulates a task that takes time to finish.

- Prints a start message, then **awaits** `asyncio.sleep(seconds)` to stand in for waiting on I/O (network, disk, etc.).
- While it waits, the event loop is free to run other tasks, so several teas can "brew" at the same time.
- Prints a done message and returns `"<name> ready"`.

**Args:** `name` (str) is the task label. `seconds` (float) is how long to wait.
**Returns:** `str`

In [1]:
import asyncio
import time

async def make_tea(name, seconds):
    print(f"Start {name}")
    await asyncio.sleep(seconds)   # simulates waiting (e.g., network/IO)
    print(f"Done {name}")
    return f"{name} ready"

### Running the teas concurrently with `asyncio.gather`

Starts all three `make_tea` tasks at once and waits for every one of them to finish.

- All three "Start" messages print right away. The "Done" messages then print in order of finish time: **Black tea (1s) → Masala chai (2s) → Green tea (3s)**.
- `results` keeps the **order the tasks were passed in**, not the order they finished:
  `['Green tea ready', 'Black tea ready', 'Masala chai ready']`
- **Total time is about 3.0s**, matching the longest task, instead of 6s if the tasks ran one after another.

In [2]:
start = time.perf_counter()

results = await asyncio.gather(
    make_tea("Green tea", 3),
    make_tea("Black tea", 1),
    make_tea("Masala chai", 2),
)

print(results)
print(f"Total time: {time.perf_counter() - start:.1f}s")

Start Green tea
Start Black tea
Start Masala chai
Done Black tea
Done Masala chai
Done Green tea
['Green tea ready', 'Black tea ready', 'Masala chai ready']
Total time: 3.0s


### Running the teas sequentially with separate `await`s

Awaits each `make_tea` call one at a time, so each task starts only after the previous one finishes.

- Messages print in strict order: **Start/Done Green tea → Start/Done Black tea → Start/Done Masala chai**.
- The result is `['Green tea ready', 'Black tea ready', 'Masala chai ready']`.
- **Total time is about 6.0s** (3 + 1 + 2), because the waits happen back to back instead of overlapping.

> **Takeaway:** Using `async`/`await` doesn't make code concurrent by itself. To get concurrency, schedule the tasks together with `asyncio.gather` (or `asyncio.create_task`), as in the previous cell.

In [3]:
start = time.perf_counter()

r1 = await make_tea("Green tea", 3)
r2 = await make_tea("Black tea", 1)
r3 = await make_tea("Masala chai", 2)

print([r1, r2, r3])
print(f"Total time: {time.perf_counter() - start:.1f}s")

Start Green tea
Done Green tea
Start Black tea
Done Black tea
Start Masala chai
Done Masala chai
['Green tea ready', 'Black tea ready', 'Masala chai ready']
Total time: 6.0s
